In [0]:
# Create widgets
dbutils.widgets.text("source_system", "Source System")
dbutils.widgets.text("table_name",  "Table Name")
dbutils.widgets.text("landing_timestamp", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Define landing path
landing_path = f"/Volumes/adb_dev_healthcare/landing/landing/{source_system}/{table_name}/{landing_timestamp}/"

# Check if path exists, exit gracefully if no new files
try:
    dbutils.fs.ls(landing_path)
    print(f"Files found at: {landing_path}")
except Exception:
    print("No new files are received.")
    dbutils.notebook.exit("No new files are received.")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
               .withColumn("insert_timestamp", current_timestamp())

In [0]:
display(updated_df)

In [0]:
updated_df.write.mode("append").format("delta").saveAsTable(f"adb_dev_healthcare.bronze.{table_name}")

In [0]:

display(spark.sql(f'select * from adb_dev_healthcare.bronze.{table_name}'))